# Agent: Observer Agent (Sensor Network)

**Purpose:** Deploy water-level sensors and publish readings every 5 seconds.

**Sensors:** 5 sensors deployed around Køge Søndre Strand (55.4506, 12.1975)

**Publishes to:** `city/flood/observer/sensor_1` through `sensor_5` topics as JSON `ObserverReading` messages

**Data:** Water level (meters) + flow rate (simulated)

In [ ]:
import time
import random
from datetime import datetime, timezone
from pathlib import Path
import sys

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, make_topic
from simulated_city.flood import ObserverReading

# Load configuration
try:
    cfg = load_config()
    print(f"MQTT Broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
    print(f"Base Topic: {cfg.mqtt.base_topic}")
except Exception as e:
    print(f"Error loading config: {e}")
    cfg = None

# Define sensor locations (around Køge Søndre Strand)
BASE_LAT, BASE_LNG = 55.4506, 12.1975
num_sensors = 5
sensors = {
    f"sensor_{i+1}": (BASE_LAT + random.uniform(-0.0005, 0.0005), 
                      BASE_LNG + random.uniform(-0.0005, 0.0005))
    for i in range(num_sensors)
}

print("\nObserver network configured:")
print(f"  Base location: Køge Søndre Strand ({BASE_LAT}, {BASE_LNG})")
print(f"  Sensors: {num_sensors}")
for sensor_id, (lat, lng) in sensors.items():
    print(f"    - {sensor_id}: ({lat:.4f}, {lng:.4f})")

In [ ]:
if cfg:
    mqtt = MqttConnector(cfg.mqtt, client_id_suffix='observer')
    mqtt.connect()
    if mqtt.wait_for_connection(timeout=5):
        print("\n✓ Connected to MQTT broker")
    else:
        print("✗ Failed to connect to MQTT broker")
        mqtt = None

In [ ]:
# Global water level (updated by Control agent indirectly through MQTT)
# Start with baseline and add noise
global_water_level = 0.2

def get_sensor_reading(sensor_id: str) -> float:
    """Generate a sensor reading with noise."""
    # Add sensor-specific noise (±0.05m)
    noise = random.uniform(-0.05, 0.05)
    return max(0, global_water_level + noise)


def publish_sensor_reading(sensor_id: str):
    """Publish water level reading from one sensor."""
    if not mqtt:
        return False
    
    water_level = get_sensor_reading(sensor_id)
    flow_rate = random.uniform(0.1, 0.5)  # Simulated flow rate
    
    reading = ObserverReading(
        sensor_id=sensor_id,
        water_level=water_level,
        flow_rate=flow_rate,
        timestamp=datetime.now(timezone.utc).isoformat()
    )
    
    topic = make_topic(cfg.mqtt, "observer", sensor_id)
    payload = reading.to_json()
    
    mqtt.client.publish(topic, payload, qos=1)
    return True

In [ ]:
print("Starting observer network loop...")
print("Polling interval: 5s")
print("Press Ctrl+C to stop\n")

reading_count = 0
try:
    while True:
        reading_count += 1
        
        # Publish reading from each sensor
        readings_str = ""
        for sensor_id in sorted(sensors.keys()):
            publish_sensor_reading(sensor_id)
            level = get_sensor_reading(sensor_id)
            readings_str += f"  {sensor_id}: {level:.2f}m"
        
        print(f"[{reading_count}] {readings_str}")
        time.sleep(5)  # Poll every 5 seconds
        
except KeyboardInterrupt:
    print("\n\n✓ Observer agent stopped by user")
finally:
    if mqtt:
        mqtt.disconnect()

In [ ]:
POLLING_INTERVAL = 5  # Publish every 5 seconds

print(f"Starting observer network loop...")
print(f"Polling interval: {POLLING_INTERVAL}s\n")

reading_count = 0
try:
    while True:
        reading_count += 1
        
        # Get current water level
        current_level = get_current_water_level()
        
        # Publish reading from each sensor
        for sensor_loc in SENSOR_LOCATIONS:
            sensor_id = sensor_loc["id"]
            reading = publish_reading(sensor_id, current_level, flow_rate=0.0)
            print(f"[{reading_count}] {sensor_id}: {reading.water_level:.2f}m", end="  ")
        
        print()  # Newline
        time.sleep(POLLING_INTERVAL)

except KeyboardInterrupt:
    print("\n\nObserver network stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

## Sensor Network Loop

Each sensor publishes readings every 5 seconds. All sensors measure the same water level
(representing a unified water body), but with independent sensor noise.

Run this cell to start the observer network (press Ctrl+C to stop).

In [ ]:
# Shared water level state (simulating a physical water body)
# In reality, all sensors would measure the same water level
water_level_state = {"level": 0.0, "lock": threading.Lock()}

def get_current_water_level():
    """Get the current water level (shared across all sensors)."""
    with water_level_state["lock"]:
        return water_level_state["level"]

def set_water_level(level):
    """Set the water level (called by external trigger simulation)."""
    with water_level_state["lock"]:
        water_level_state["level"] = max(0, level)

def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_reading(sensor_id, water_level, flow_rate=0.0):
    """Publish a water level reading from a sensor."""
    # Add small sensor noise
    noisy_level = water_level + random.uniform(-0.2, 0.2)
    noisy_level = max(0, noisy_level)  # Water level can't be negative
    
    reading = ObserverReading(
        sensor_id=sensor_id,
        water_level=noisy_level,
        flow_rate=flow_rate,
        timestamp=get_iso_timestamp()
    )
    
    topic = make_topic(mqtt_cfg, "observer", sensor_id)
    payload = json.dumps(reading.to_json())
    publisher.publish_json(topic, payload, qos=1)
    
    return reading

print("Water level simulation configured.")

## Water Level Simulation

Model simulates a realistic flood cycle:
- Baseline: 0-0.5m (normal conditions)
- Flood phase: ~6-7m (triggered by external event)
- Recovery: back to baseline

Each sensor adds small noise ±0.2m to simulate realistic variations.

In [ ]:
# Connect to MQTT
connector = MqttConnector(mqtt_cfg, client_id_suffix="observer")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

## Connect to MQTT Broker

In [ ]:
import time
import json
import random
from datetime import datetime, timezone
import threading

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import ObserverReading

# Load configuration
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

# Sensor network configuration
KOEGE_STRAND_LAT = 55.45058843620187
KOEGE_STRAND_LON = 12.197503222443261
NUM_SENSORS = 5
SENSOR_SPACING_KM = 0.002  # ~200 meters spacing

# Create sensor locations (small grid around Køge Søndre Strand)
SENSOR_LOCATIONS = []
for i in range(NUM_SENSORS):
    lat = KOEGE_STRAND_LAT + (i % 3) * SENSOR_SPACING_KM / 111  # 111 km per degree
    lon = KOEGE_STRAND_LON + (i // 3) * SENSOR_SPACING_KM / 111
    SENSOR_LOCATIONS.append({
        "id": f"sensor_{i+1}",
        "lat": lat,
        "lon": lon
    })

print(f"Observer network configured:")
print(f"  Base location: Køge Søndre Strand ({KOEGE_STRAND_LAT}, {KOEGE_STRAND_LON})")
print(f"  Sensors: {NUM_SENSORS}")
for loc in SENSOR_LOCATIONS:
    print(f"    - {loc['id']}: ({loc['lat']:.4f}, {loc['lon']:.4f})")

## Setup and Configuration

# Agent: Observer (Water Level Sensors)

Sensor network that monitors water levels near Køge Søndre Strand (beach).

**Setup:**
- 5 sensors deployed near Køge Søndre Strand
- Each publishes water level readings every 5 seconds
- Water rises to 6-7m during flood, recovers afterward
- Simulates realistic sensor variability with small noise

**Coordinates:** All sensors centered around 55.4506, 12.1975 (Køge Søndre Strand)

# Agent Observer
Sensor notebook that reads water levels and publishes `ObserverReading` messages.
Can be instantiated multiple times with different IDs.